In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import shutil
from PIL import Image
from sklearn.model_selection import train_test_split

# Chemins
chemin_images = '/content/drive/MyDrive/Project/merged_dataset'
chemin_csv = '/content/drive/MyDrive/Project/updated_all_annotation.csv'
chemin_yolo = '/content/drive/MyDrive/Project/yolo_dataset'

# Créer les dossiers
for split in ['train', 'val']:
    os.makedirs(f'{chemin_yolo}/images/{split}', exist_ok=True)
    os.makedirs(f'{chemin_yolo}/labels/{split}', exist_ok=True)

# Lire le CSV
annotations = pd.read_csv(chemin_csv)

# Diviser aléatoirement en train/val (80% / 20%)
train_ann, val_ann = train_test_split(annotations, test_size=0.2, random_state=42)


In [ ]:
import os

# Ensure directories exist
os.makedirs(f'{chemin_yolo}/images/train', exist_ok=True)
os.makedirs(f'{chemin_yolo}/images/val', exist_ok=True)
os.makedirs(f'{chemin_yolo}/labels/train', exist_ok=True)
os.makedirs(f'{chemin_yolo}/labels/val', exist_ok=True)



In [ ]:
def convert_to_yolo_format(xmin, ymin, xmax, ymax, img_w, img_h):
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h
    return x_center, y_center, width, height


In [ ]:
def process_annotations(df, split):
    for _, row in df.iterrows():
        filename = row['filename']
        img_path = os.path.join(chemin_images, filename)

        try:
            img = Image.open(img_path)
            w, h = img.size
        except:
            continue

        x_center, y_center, width, height = convert_to_yolo_format(
            row['xmin'], row['ymin'], row['xmax'], row['ymax'], w, h
        )

        # Copier image
        shutil.copy(img_path, f'{chemin_yolo}/images/{split}/{filename}')

        # Créer fichier label
        txt_name = os.path.splitext(filename)[0] + '.txt'
        with open(f'{chemin_yolo}/labels/{split}/{txt_name}', 'w') as f:
            f.write(f'0 {x_center} {y_center} {width} {height}\n')

process_annotations(train_ann, 'train')
process_annotations(val_ann, 'val')


In [ ]:
data_yaml = """
train: /content/drive/MyDrive/Project/yolo_dataset/images/train
val: /content/drive/MyDrive/Project/yolo_dataset/images/val

nc: 1
names: ['license_plate']
"""


In [ ]:
# Specify the path where you want to save the YAML file
file_path = '/content/drive/MyDrive/Project/yolo_dataset/data.yaml'

# Open the file and write the data to it
with open(file_path, 'w') as file:
    file.write(data_yaml)

print(f"Data YAML file saved at: {file_path}")


In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

In [ ]:
!python train.py --img 640 --batch 16 --epochs 25 --data /content/drive/MyDrive/Project/yolo_dataset/data.yaml --weights yolov5s.pt --project /content/drive/MyDrive/Project/yolo_training --name lp_detector